# Ball Walker — SAC Training on Google Colab

**執行前先確認：**
- 上方選單 `執行階段 → 變更執行階段類型` → GPU (T4 即可)
- 若要把 checkpoint 存到 Google Drive，執行 Cell 2 掛載 Drive；否則跳過

**訓練完成後** Cell 7 會自動把 `models/` 壓縮成 `ball_walker_model.zip` 並觸發瀏覽器下載。

## Step 1 — 安裝依賴套件

In [1]:
# 安裝所有依賴；Colab 重啟 Runtime 後需重新執行此 Cell
!pip install --quiet \
    "stable-baselines3[extra]==2.3.2" \
    "gymnasium==0.29.1" \
    "pybullet==3.2.6" \
    "tensorboard>=2.14" \
    "tqdm>=4.66"

print("✅ 套件安裝完成")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.5/80.5 MB 10.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 34.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: Could not find a version that satisfies the requirement ale-py~=0.8.1; extra == "atari" (from shimmy[atari]) (from versions: 0.9.0, 0.9.1, 0.10.0, 0.10.1, 0.10.2, 0.11.0, 0.11.1, 0.11.2)
ERROR: No matching distribution found for ale-py~=0.8.1; extra == "atari"
✅ 套件安裝完成


## Step 2 — （可選）掛載 Google Drive

掛載後 checkpoint 會存到 `MyDrive/ball_walker_checkpoints/`，斷線後不會遺失。
**不需要 Drive 的話直接跳到 Step 3。**

In [ ]:
USE_DRIVE = True  # 改成 False 可跳過

DRIVE_MODEL_DIR = "/content/drive/MyDrive/ball_walker_checkpoints"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    import os
    os.makedirs(DRIVE_MODEL_DIR, exist_ok=True)
    print(f"✅ Drive 掛載完成，checkpoint 將儲存至 {DRIVE_MODEL_DIR}")
else:
    print("⏭️  跳過 Drive 掛載，checkpoint 儲存於本機 /content/models/")

## Step 3 — Clone 專案（若尚未執行）

In [ ]:
import os, sys

REPO_URL  = "https://github.com/DioWang67/pybullet_train.git"  # ← 修改為你的 repo URL
REPO_DIR  = "/content/pybullet_walking"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print("Repo 已存在，執行 git pull ...")
    !git -C {REPO_DIR} pull

# 把專案根目錄加入 Python path，讓 `from envs.ball_walker_env import ...` 可以找到
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

os.chdir(REPO_DIR)
print(f"✅ 工作目錄切換至 {os.getcwd()}")

## Step 4 — 驗證環境可正常初始化（headless）

In [ ]:
from stable_baselines3.common.env_checker import check_env
from envs.ball_walker_env import BallWalkerEnv

# render_mode=None → PyBullet 使用 DIRECT 模式，Colab 無 GUI 也能執行
test_env = BallWalkerEnv(render_mode=None)
check_env(test_env, warn=True)
test_env.close()
print("✅ 環境驗證通過")

## Step 5 — 設定訓練參數

In [ ]:
import os

# ── 路徑：checkpoint 是否存到 Drive ──────────────────────────────────────────
try:
    _use_drive = USE_DRIVE
except NameError:
    _use_drive = False

if _use_drive and os.path.isdir("/content/drive/MyDrive"):
    MODEL_DIR = "/content/drive/MyDrive/ball_walker_checkpoints"
else:
    MODEL_DIR = "/content/models"

LOG_DIR   = "/content/logs/ball_walker"
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOG_DIR,   exist_ok=True)

MODEL_PATH   = os.path.join(MODEL_DIR, "ball_walker_sac")
VECNORM_PATH = os.path.join(MODEL_DIR, "ball_walker_vecnorm.pkl")

# ── 訓練超參數 ────────────────────────────────────────────────────────────────
# Colab 免費版單核 CPU，SubprocVecEnv 效益有限；使用 DummyVecEnv 避免 fork 問題
N_ENVS          = 4
TOTAL_TIMESTEPS = 2_000_000   # 可依需求調整；5M 約需 2-3 小時
EVAL_FREQ       = 25_000
CHECKPOINT_FREQ = 50_000

print(f"Model dir  : {MODEL_DIR}")
print(f"Log dir    : {LOG_DIR}")
print(f"Timesteps  : {TOTAL_TIMESTEPS:,}")
print(f"Parallel envs: {N_ENVS}")

## Step 6 — 開始訓練

In [ ]:
import time
from collections import deque

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    BaseCallback,
    CheckpointCallback,
    EvalCallback,
)
from envs.ball_walker_env import BallWalkerEnv


# ── Callbacks（與 train_ball_walker_v2.py 相同邏輯）─────────────────────────
class SyncVecNormCallback(BaseCallback):
    """每次 eval 前將訓練 env 的 obs/ret 統計同步到 eval env。"""
    def __init__(self, train_env: VecNormalize, eval_env: VecNormalize):
        super().__init__(verbose=0)
        self.train_env = train_env
        self.eval_env  = eval_env

    def _on_step(self) -> bool:
        self.eval_env.obs_rms = self.train_env.obs_rms
        self.eval_env.ret_rms = self.train_env.ret_rms
        return True


class TrainingProgressCallback(BaseCallback):
    """每 10k steps 印一行訓練狀態。"""
    PRINT_FREQ = 10_000

    def __init__(self, total_timesteps: int):
        super().__init__(verbose=0)
        self.total_timesteps = total_timesteps
        self._ep_rewards: deque = deque(maxlen=20)
        self._ep_lengths: deque = deque(maxlen=20)
        self._best_reward = -np.inf
        self._n_episodes  = 0
        self._t_start     = 0.0
        self._last_print  = 0

    def _on_training_start(self) -> None:
        self._t_start = time.time()
        print(f"\n{'Steps':>12}  {'Episodes':>8}  {'Mean R(20)':>10}  "
              f"{'Best R':>9}  {'FPS':>5}  {'ETA':>9}")
        print("-" * 62)

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            ep = info.get("episode")
            if ep is not None:
                self._ep_rewards.append(ep["r"])
                self._ep_lengths.append(ep["l"])
                self._n_episodes += 1
                if ep["r"] > self._best_reward:
                    self._best_reward = ep["r"]

        if self.num_timesteps - self._last_print >= self.PRINT_FREQ:
            self._last_print = self.num_timesteps
            elapsed   = time.time() - self._t_start
            fps       = int(self.num_timesteps / elapsed) if elapsed > 0 else 0
            remaining = (self.total_timesteps - self.num_timesteps) / fps if fps > 0 else 0
            mean_r    = np.mean(self._ep_rewards) if self._ep_rewards else float("nan")
            best_r    = self._best_reward if self._ep_rewards else float("nan")
            mins, secs = divmod(int(remaining), 60)
            hrs,  mins = divmod(mins, 60)
            print(f"{self.num_timesteps:>12,}  {self._n_episodes:>8}  "
                  f"{mean_r:>10.1f}  {best_r:>9.1f}  "
                  f"{fps:>5}  {hrs:02d}:{mins:02d}:{secs:02d}")
        return True

    def _on_training_end(self) -> None:
        elapsed = time.time() - self._t_start
        m, s = divmod(int(elapsed), 60)
        h, m = divmod(m, 60)
        print("-" * 62)
        print(f"Done! Total time {h:02d}:{m:02d}:{s:02d}  "
              f"Best reward = {self._best_reward:.1f}  "
              f"Episodes = {self._n_episodes}")


# ── 環境工廠（全部 headless，Colab 無 GUI）──────────────────────────────────
def _make_env():
    return Monitor(BallWalkerEnv(render_mode=None))


# ── 建立向量化環境 ────────────────────────────────────────────────────────────
# 注意：Colab 使用 DummyVecEnv（而非 SubprocVecEnv）
# 原因：Colab 的 multiprocessing fork 在部分版本會 hang；
#       DummyVecEnv 在同一 process 內串行執行，穩定性更高。
vec_env = DummyVecEnv([_make_env] * N_ENVS)
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

eval_vec = DummyVecEnv([_make_env])
eval_vec = VecNormalize(
    eval_vec,
    norm_obs=True,
    norm_reward=False,
    clip_obs=10.0,
    training=False,
)

# ── SAC 模型 ──────────────────────────────────────────────────────────────────
model = SAC(
    "MlpPolicy",
    vec_env,
    learning_rate=3e-4,
    buffer_size=500_000,   # Colab RAM 有限，縮小 replay buffer
    batch_size=256,
    tau=0.005,
    gamma=0.99,
    train_freq=1,
    gradient_steps=1,
    policy_kwargs={"net_arch": [256, 256]},
    verbose=0,             # 由 TrainingProgressCallback 輸出進度
    tensorboard_log=LOG_DIR,
    device="auto",         # 有 GPU 自動使用
)

# ── Callbacks ─────────────────────────────────────────────────────────────────
checkpoint_cb = CheckpointCallback(
    save_freq=max(CHECKPOINT_FREQ // N_ENVS, 1),
    save_path=MODEL_DIR,
    name_prefix="ball_walker",
    verbose=0,
)
sync_cb     = SyncVecNormCallback(train_env=vec_env, eval_env=eval_vec)
progress_cb = TrainingProgressCallback(total_timesteps=TOTAL_TIMESTEPS)
eval_cb = EvalCallback(
    eval_vec,
    best_model_save_path=MODEL_DIR,
    log_path=LOG_DIR,
    eval_freq=max(EVAL_FREQ // N_ENVS, 1),
    n_eval_episodes=5,
    deterministic=True,
    verbose=0,
)

# ── 開始訓練 ──────────────────────────────────────────────────────────────────
print(f"Training for {TOTAL_TIMESTEPS:,} steps with {N_ENVS} envs (DummyVecEnv) ...")
model.learn(
    total_timesteps=TOTAL_TIMESTEPS,
    callback=[sync_cb, progress_cb, checkpoint_cb, eval_cb],
    progress_bar=False,  # tqdm progress bar 在 Colab 顯示效果差
)

# ── 儲存最終模型 ──────────────────────────────────────────────────────────────
model.save(MODEL_PATH)
vec_env.save(VECNORM_PATH)
print(f"\nModel saved      → {MODEL_PATH}.zip")
print(f"VecNormalize saved → {VECNORM_PATH}")

vec_env.close()
eval_vec.close()

## Step 7 — 下載訓練好的模型到本機

執行後瀏覽器會自動下載 `ball_walker_model.zip`。
解壓後把 `models/` 資料夾放到本機專案根目錄，即可執行：
```
python train_ball_walker_v2.py --render
```

In [ ]:
import os, shutil
from google.colab import files

# 打包 MODEL_DIR 整個資料夾成 zip
DOWNLOAD_ZIP = "/content/ball_walker_model.zip"

# shutil.make_archive(output_path_without_ext, format, root_dir, base_dir)
archive_base = DOWNLOAD_ZIP.replace(".zip", "")
shutil.make_archive(
    base_name=archive_base,
    format="zip",
    root_dir=os.path.dirname(MODEL_DIR),
    base_dir=os.path.basename(MODEL_DIR),
)

zip_size_mb = os.path.getsize(DOWNLOAD_ZIP) / (1024 ** 2)
print(f"Archive size: {zip_size_mb:.1f} MB")
print("Downloading...")
files.download(DOWNLOAD_ZIP)

## Step 8 — （可選）TensorBoard 可視化訓練曲線

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/logs

## Step 9 — （可選）從現有 checkpoint 繼續訓練

若 Colab 斷線後想繼續，從 Drive 或重新上傳 checkpoint，再執行此 Cell。

In [ ]:
import os, glob
from stable_baselines3 import SAC
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
from envs.ball_walker_env import BallWalkerEnv

# 找最新的 checkpoint
checkpoints = sorted(glob.glob(os.path.join(MODEL_DIR, "ball_walker_*_steps.zip")))
if not checkpoints:
    print("找不到 checkpoint，請先執行 Step 6")
else:
    latest_ckpt = checkpoints[-1]
    print(f"載入 checkpoint: {latest_ckpt}")

    def _make_env():
        return Monitor(BallWalkerEnv(render_mode=None))

    vec_env = DummyVecEnv([_make_env] * N_ENVS)

    if os.path.exists(VECNORM_PATH):
        vec_env = VecNormalize.load(VECNORM_PATH, vec_env)
        vec_env.training = True
        vec_env.norm_reward = True
        print(f"VecNormalize 統計載入自 {VECNORM_PATH}")
    else:
        vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

    model = SAC.load(latest_ckpt, env=vec_env, device="auto")

    RESUME_TIMESTEPS = 1_000_000   # 繼續訓練的額外步數
    print(f"繼續訓練 {RESUME_TIMESTEPS:,} steps ...")
    model.learn(
        total_timesteps=RESUME_TIMESTEPS,
        reset_num_timesteps=False,   # 保留原步數計數
    )
    model.save(MODEL_PATH)
    vec_env.save(VECNORM_PATH)
    print("儲存完成")
    vec_env.close()